# Hyperparameter Tuning Notebook (MNIST + CIFAR-10)

This notebook runs a **10-combination hyperparameter search** across three MLP architectures:
- Shallow: `[128]`
- Medium: `[512, 256, 128]`
- Deep: `[1024, 512, 256, 128, 64]`

It uses your project modules:
- `dataset_loaders.py`
- `model.py`
- `train.py`

Preprocessing in `dataset_loaders.py` now applies `ToTensor()` plus dataset-specific standard normalization for both MNIST and CIFAR-10.

In [1]:
!git clone https://github.com/bnguyen1212/CS4375-Project3-BHN220001.git
%cd CS4375-Project3-BHN220001

Cloning into 'CS4375-Project3-BHN220001'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 29 (delta 14), reused 18 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 21.04 KiB | 3.01 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/CS4375-Project3-BHN220001


In [2]:
# If needed, run this once in notebook runtime:
# %pip install -r requirements.txt

import os
import sys
import random
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import ConcatDataset, DataLoader

try:
    from dataset_loaders import load_mnist, load_cifar10
    from model import MLP, SimpleCNN, EnhancedCNN
    from train import train_model, evaluate
except ModuleNotFoundError as e:
    print(f'Import error: {e}')
    print(f'Current working directory: {os.getcwd()}')
    print('Top sys.path entries:')
    for p in sys.path[:5]:
        print(' -', p)
    print('Hint: run Cell 2 first and ensure it prints the correct project root.')
    raise

In [3]:
# Reproducibility + device
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
# Experiment settings
EPOCHS = 20
PATIENCE = 5
NUM_WORKERS = 0

CNN_ARCHITECTURES = ["simple_cnn", "enhanced_cnn"]
CNN_DROPOUT = 0.2

ARCHITECTURES = {
    "shallow": [128],
    "medium": [512, 256, 128],
    "deep": [1024, 512, 256, 128, 64],
}

# 10 meaningful CNN tuning combinations from:
# LR {0.01, 0.001, 0.0001}
# Batch size {32, 64, 128}
# Weight decay {0, 1e-4, 5e-3}
CNN_SEARCH_CONFIGS = [
    {"lr": 1e-2, "batch_size": 32,  "weight_decay": 0.0,  "dropout": CNN_DROPOUT},
    {"lr": 1e-2, "batch_size": 64,  "weight_decay": 1e-4, "dropout": CNN_DROPOUT},
    {"lr": 1e-2, "batch_size": 128, "weight_decay": 5e-3, "dropout": CNN_DROPOUT},
    {"lr": 1e-3, "batch_size": 32,  "weight_decay": 1e-4, "dropout": CNN_DROPOUT},
    {"lr": 1e-3, "batch_size": 64,  "weight_decay": 0.0,  "dropout": CNN_DROPOUT},
    {"lr": 1e-3, "batch_size": 128, "weight_decay": 5e-3, "dropout": CNN_DROPOUT},
    {"lr": 1e-4, "batch_size": 32,  "weight_decay": 5e-3, "dropout": CNN_DROPOUT},
    {"lr": 1e-4, "batch_size": 64,  "weight_decay": 1e-4, "dropout": CNN_DROPOUT},
    {"lr": 1e-4, "batch_size": 128, "weight_decay": 0.0,  "dropout": CNN_DROPOUT},
    {"lr": 1e-3, "batch_size": 64,  "weight_decay": 5e-3, "dropout": CNN_DROPOUT},
]

# 10 meaningful combinations from:
# LR {0.01, 0.001, 0.0001}
# Batch size {32, 64, 128}
# Optimizer {sgd, adam}
# Dropout {0.2, 0.5}
SEARCH_CONFIGS = [
    {"lr": 1e-2,  "batch_size": 64,  "optimizer": "sgd",  "dropout": 0.2},
    {"lr": 1e-3,  "batch_size": 32,  "optimizer": "adam", "dropout": 0.2},
    {"lr": 1e-4,  "batch_size": 128, "optimizer": "adam", "dropout": 0.5},
    {"lr": 1e-2,  "batch_size": 128, "optimizer": "sgd",  "dropout": 0.5},
    {"lr": 1e-3,  "batch_size": 64,  "optimizer": "sgd",  "dropout": 0.2},
    {"lr": 1e-4,  "batch_size": 32,  "optimizer": "sgd",  "dropout": 0.2},
    {"lr": 1e-3,  "batch_size": 128, "optimizer": "adam", "dropout": 0.5},
    {"lr": 1e-2,  "batch_size": 32,  "optimizer": "adam", "dropout": 0.2},
    {"lr": 1e-4,  "batch_size": 64,  "optimizer": "adam", "dropout": 0.2},
    {"lr": 1e-3,  "batch_size": 64,  "optimizer": "sgd",  "dropout": 0.5},
]

print(f"Architectures: {list(ARCHITECTURES.keys())}")
print(f"#search configs: {len(SEARCH_CONFIGS)}")
print(f"CNN architectures: {CNN_ARCHITECTURES}")
print(f"#cnn search configs: {len(CNN_SEARCH_CONFIGS)}")

Architectures: ['shallow', 'medium', 'deep']
#search configs: 10
CNN architectures: ['simple_cnn', 'enhanced_cnn']
#cnn search configs: 10


In [5]:
# Helpers
def get_loaders(dataset_name: str, batch_size: int):
    if dataset_name == "mnist":
        return load_mnist(batch_size=batch_size, num_workers=NUM_WORKERS, split_seed=SEED)
    if dataset_name == "cifar10":
        return load_cifar10(batch_size=batch_size, num_workers=NUM_WORKERS, split_seed=SEED)
    raise ValueError("dataset_name must be 'mnist' or 'cifar10'")

def build_mlp(dataset_name: str, hidden_dims, dropout: float):
    if dataset_name == "mnist":
        input_dim = 28 * 28
    elif dataset_name == "cifar10":
        input_dim = 32 * 32 * 3
    else:
        raise ValueError("dataset_name must be 'mnist' or 'cifar10'")

    return MLP(input_dim=input_dim, hidden_dims=hidden_dims, dropout=dropout, num_classes=10)

def build_cnn(dataset_name: str, architecture: str, dropout: float):
    if dataset_name == "mnist":
        in_channels = 1
    elif dataset_name == "cifar10":
        in_channels = 3
    else:
        raise ValueError("dataset_name must be 'mnist' or 'cifar10'")

    if architecture == "simple_cnn":
        return SimpleCNN(in_channels=in_channels, dropout=dropout, num_classes=10, filters=8)
    if architecture == "enhanced_cnn":
        return EnhancedCNN(in_channels=in_channels, dropout=dropout, num_classes=10, filters=(16, 32, 64))

    raise ValueError("architecture must be 'simple_cnn' or 'enhanced_cnn'")

In [6]:
# Main tuning loop split by dataset so each run can be executed in separate cells
all_results = []
best_models = {}
best_histories = {}
run_idx = 0
total_runs = 2 * len(ARCHITECTURES) * len(SEARCH_CONFIGS)

def run_tuning_for_dataset(dataset_name):
    global run_idx

    for arch_name, hidden_dims in ARCHITECTURES.items():
        best_key = (dataset_name, arch_name)
        best_val_acc = -1.0
        best_model = None
        best_history = None

        for cfg_i, cfg in enumerate(SEARCH_CONFIGS, start=1):
            run_idx += 1
            print("=" * 90)
            print(
                f"Run {run_idx}/{total_runs} | dataset={dataset_name} | arch={arch_name} "
                f"| cfg={cfg_i}/10 | {cfg}"
            )

            train_loader, val_loader, _ = get_loaders(dataset_name, cfg["batch_size"])
            model = build_mlp(dataset_name, hidden_dims, dropout=cfg["dropout"])

            train_cfg = {
                "lr": cfg["lr"],
                "optimizer": cfg["optimizer"],
                "epochs": EPOCHS,
                "patience": PATIENCE,
            }

            trained_model, history, runtime_sec = train_model(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
                config=train_cfg,
                device=device,
            )

            final_val_acc = history["val_acc"][-1] if history["val_acc"] else 0.0
            best_epoch_val_acc = max(history["val_acc"]) if history["val_acc"] else 0.0

            all_results.append({
                "dataset": dataset_name,
                "architecture": arch_name,
                "hidden_dims": str(hidden_dims),
                "config_id": cfg_i,
                "lr": cfg["lr"],
                "batch_size": cfg["batch_size"],
                "optimizer": cfg["optimizer"],
                "dropout": cfg["dropout"],
                "epochs_ran": len(history["train_loss"]),
                "final_train_loss": history["train_loss"][-1] if history["train_loss"] else None,
                "final_val_loss": history["val_loss"][-1] if history["val_loss"] else None,
                "final_val_acc": final_val_acc,
                "best_val_acc": best_epoch_val_acc,
                "runtime_sec": runtime_sec,
            })

            if best_epoch_val_acc > best_val_acc:
                best_val_acc = best_epoch_val_acc
                best_model = deepcopy(trained_model)
                best_history = deepcopy(history)

        best_models[best_key] = best_model
        best_histories[best_key] = best_history

    print(f"\nCompleted tuning for {dataset_name}.")

print("Run the next two cells separately: one for MNIST and one for CIFAR-10.")

# Separate CNN tuning loop
cnn_all_results = []
cnn_best_models = {}
cnn_best_histories = {}
cnn_run_idx = 0
cnn_total_runs = 2 * len(CNN_ARCHITECTURES) * len(CNN_SEARCH_CONFIGS)

def run_cnn_tuning_for_dataset(dataset_name):
    global cnn_run_idx

    for arch_name in CNN_ARCHITECTURES:
        best_key = (dataset_name, arch_name)
        best_val_acc = -1.0
        best_model = None
        best_history = None

        for cfg_i, cfg in enumerate(CNN_SEARCH_CONFIGS, start=1):
            cnn_run_idx += 1
            print("=" * 90)
            print(
                f"CNN Run {cnn_run_idx}/{cnn_total_runs} | dataset={dataset_name} | arch={arch_name} "
                f"| cfg={cfg_i}/10 | {cfg}"
            )

            train_loader, val_loader, _ = get_loaders(dataset_name, cfg["batch_size"])
            model = build_cnn(dataset_name, arch_name, dropout=cfg["dropout"])

            train_cfg = {
                "lr": cfg["lr"],
                "optimizer": "adam",
                "weight_decay": cfg["weight_decay"],
                "epochs": EPOCHS,
                "patience": PATIENCE,
            }

            trained_model, history, runtime_sec = train_model(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
                config=train_cfg,
                device=device,
            )

            final_val_acc = history["val_acc"][-1] if history["val_acc"] else 0.0
            best_epoch_val_acc = max(history["val_acc"]) if history["val_acc"] else 0.0

            cnn_all_results.append({
                "dataset": dataset_name,
                "architecture": arch_name,
                "config_id": cfg_i,
                "lr": cfg["lr"],
                "batch_size": cfg["batch_size"],
                "optimizer": "adam",
                "weight_decay": cfg["weight_decay"],
                "dropout": cfg["dropout"],
                "epochs_ran": len(history["train_loss"]),
                "final_train_loss": history["train_loss"][-1] if history["train_loss"] else None,
                "final_val_loss": history["val_loss"][-1] if history["val_loss"] else None,
                "final_val_acc": final_val_acc,
                "best_val_acc": best_epoch_val_acc,
                "runtime_sec": runtime_sec,
            })

            if best_epoch_val_acc > best_val_acc:
                best_val_acc = best_epoch_val_acc
                best_model = deepcopy(trained_model)
                best_history = deepcopy(history)

        cnn_best_models[best_key] = best_model
        cnn_best_histories[best_key] = best_history

    print(f"\nCompleted CNN tuning for {dataset_name}.")

print("Run the next two cells for CNN tuning: one for MNIST and one for CIFAR-10.")

Run the next two cells separately: one for MNIST and one for CIFAR-10.
Run the next two cells for CNN tuning: one for MNIST and one for CIFAR-10.


In [ ]:
# Run MLP tuning for MNIST only
run_tuning_for_dataset("mnist")

In [ ]:
# Run MLP tuning for CIFAR-10 only
run_tuning_for_dataset("cifar10")

print("\nMLP tuning loop complete.")

In [ ]:
# Summarize MLP tuning runs and keep only the best config per dataset/architecture
mlp_results_df = pd.DataFrame(all_results)

mlp_best_cfg_df = (
    mlp_results_df
    .sort_values("best_val_acc", ascending=False)
    .groupby(["dataset", "architecture"], as_index=False)
    .first()
    .sort_values(["dataset", "architecture"])
    .reset_index(drop=True)
)

print(f"Total completed MLP runs: {len(mlp_results_df)}")
print("Best MLP validation config per dataset+architecture:")
display(
    mlp_best_cfg_df[[
        "dataset", "architecture", "lr", "batch_size",
        "optimizer", "dropout", "best_val_acc", "epochs_ran"
    ]]
)

In [ ]:
# Retrain best MLP configs on full train+validation, then evaluate on test
criterion = nn.CrossEntropyLoss()
mlp_retrained_models = {}
mlp_test_rows = []

for _, row in mlp_best_cfg_df.iterrows():
    dataset_name = row["dataset"]
    arch_name = row["architecture"]
    cfg = {
        "lr": float(row["lr"]),
        "batch_size": int(row["batch_size"]),
        "optimizer": str(row["optimizer"]),
        "dropout": float(row["dropout"]),
    }

    train_loader, val_loader, test_loader = get_loaders(dataset_name, cfg["batch_size"])
    full_train_dataset = ConcatDataset([train_loader.dataset, val_loader.dataset])
    full_train_loader = DataLoader(
        full_train_dataset,
        batch_size=cfg["batch_size"],
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    hidden_dims = ARCHITECTURES[arch_name]
    model = build_mlp(dataset_name, hidden_dims, dropout=cfg["dropout"])
    retrain_cfg = {
        "lr": cfg["lr"],
        "optimizer": cfg["optimizer"],
        "epochs": EPOCHS,
    }

    retrained_model, retrain_history, retrain_runtime_sec = train_model(
        model=model,
        train_loader=full_train_loader,
        val_loader=full_train_loader,
        config=retrain_cfg,
        device=device,
    )

    test_loss, test_acc = evaluate(retrained_model, test_loader, criterion, device)
    mlp_retrained_models[(dataset_name, arch_name)] = retrained_model

    mlp_test_rows.append({
        "dataset": dataset_name,
        "architecture": arch_name,
        "lr": cfg["lr"],
        "batch_size": cfg["batch_size"],
        "optimizer": cfg["optimizer"],
        "dropout": cfg["dropout"],
        "best_val_acc": float(row["best_val_acc"]),
        "retrain_epochs": len(retrain_history["train_loss"]),
        "retrain_runtime_sec": retrain_runtime_sec,
        "test_loss": test_loss,
        "test_acc": test_acc,
    })

mlp_test_df = pd.DataFrame(mlp_test_rows).sort_values(["dataset", "architecture"]).reset_index(drop=True)
print("Final MLP test performance after full train+validation retraining:")
display(mlp_test_df)

In [ ]:
# Save MLP artifacts
os.makedirs("results", exist_ok=True)
mlp_results_path = os.path.join("results", "mlp_all_tuning_results.csv")
mlp_best_cfg_path = os.path.join("results", "mlp_best_validation_configs.csv")
mlp_test_path = os.path.join("results", "mlp_best_models_test_results.csv")

mlp_results_df.to_csv(mlp_results_path, index=False)
mlp_best_cfg_df.to_csv(mlp_best_cfg_path, index=False)
mlp_test_df.to_csv(mlp_test_path, index=False)

print("Saved:")
print(f"- {mlp_results_path}")
print(f"- {mlp_best_cfg_path}")
print(f"- {mlp_test_path}")

In [7]:
# Run CNN tuning for MNIST only
run_cnn_tuning_for_dataset("mnist")

CNN Run 1/40 | dataset=mnist | arch=simple_cnn | cfg=1/10 | {'lr': 0.01, 'batch_size': 32, 'weight_decay': 0.0, 'dropout': 0.2}


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.10MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 134kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.27MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.41MB/s]


Epoch 1/20 | train_loss=1.5109 | val_loss=0.9623 | val_acc=0.7022
Epoch 2/20 | train_loss=1.0996 | val_loss=0.7496 | val_acc=0.8042
Epoch 3/20 | train_loss=0.9726 | val_loss=0.5797 | val_acc=0.8608
Epoch 4/20 | train_loss=0.8717 | val_loss=0.5255 | val_acc=0.8679
Epoch 5/20 | train_loss=0.8130 | val_loss=0.5010 | val_acc=0.8659
Epoch 6/20 | train_loss=0.7776 | val_loss=0.4430 | val_acc=0.8807
Epoch 7/20 | train_loss=0.7446 | val_loss=0.4192 | val_acc=0.8882
Epoch 8/20 | train_loss=0.7237 | val_loss=0.4009 | val_acc=0.8894
Epoch 9/20 | train_loss=0.7160 | val_loss=0.3983 | val_acc=0.8917
Epoch 10/20 | train_loss=0.7083 | val_loss=0.3968 | val_acc=0.8921
Epoch 11/20 | train_loss=0.6995 | val_loss=0.3730 | val_acc=0.9039
Epoch 12/20 | train_loss=0.6964 | val_loss=0.3554 | val_acc=0.9053
Epoch 13/20 | train_loss=0.6879 | val_loss=0.3470 | val_acc=0.9095
Epoch 14/20 | train_loss=0.6843 | val_loss=0.3557 | val_acc=0.9070
Epoch 15/20 | train_loss=0.6774 | val_loss=0.3403 | val_acc=0.9091
Epoc

In [7]:
# Run CNN tuning for CIFAR-10 only
run_cnn_tuning_for_dataset("cifar10")

print("\nCNN tuning loop complete.")

CNN Run 1/40 | dataset=cifar10 | arch=simple_cnn | cfg=1/10 | {'lr': 0.01, 'batch_size': 32, 'weight_decay': 0.0, 'dropout': 0.2}


100%|██████████| 170M/170M [00:04<00:00, 42.6MB/s]


Epoch 1/20 | train_loss=1.9201 | val_loss=1.7867 | val_acc=0.3354
Epoch 2/20 | train_loss=1.8112 | val_loss=1.6682 | val_acc=0.3748
Epoch 3/20 | train_loss=1.7656 | val_loss=1.6332 | val_acc=0.3998
Epoch 4/20 | train_loss=1.7416 | val_loss=1.5712 | val_acc=0.4204
Epoch 5/20 | train_loss=1.7227 | val_loss=1.5803 | val_acc=0.4186
Epoch 6/20 | train_loss=1.7142 | val_loss=1.5961 | val_acc=0.4054
Epoch 7/20 | train_loss=1.7100 | val_loss=1.5353 | val_acc=0.4476
Epoch 8/20 | train_loss=1.7005 | val_loss=1.5399 | val_acc=0.4480
Epoch 9/20 | train_loss=1.6865 | val_loss=1.5340 | val_acc=0.4436
Epoch 10/20 | train_loss=1.6919 | val_loss=1.5676 | val_acc=0.4228
Epoch 11/20 | train_loss=1.6885 | val_loss=1.5239 | val_acc=0.4376
Epoch 12/20 | train_loss=1.6824 | val_loss=1.5282 | val_acc=0.4482
Epoch 13/20 | train_loss=1.6773 | val_loss=1.5283 | val_acc=0.4356
Epoch 14/20 | train_loss=1.6829 | val_loss=1.5334 | val_acc=0.4412
Epoch 15/20 | train_loss=1.6722 | val_loss=1.5319 | val_acc=0.4370
Epoc

In [8]:
# Summarize CNN tuning runs and keep only the best config per dataset/architecture
cnn_results_df = pd.DataFrame(cnn_all_results)

cnn_best_cfg_df = (
    cnn_results_df
    .sort_values("best_val_acc", ascending=False)
    .groupby(["dataset", "architecture"], as_index=False)
    .first()
    .sort_values(["dataset", "architecture"])
    .reset_index(drop=True)
)

print(f"Total completed CNN runs: {len(cnn_results_df)}")
print("Best CNN validation config per dataset+architecture:")
display(
    cnn_best_cfg_df[[
        "dataset", "architecture", "lr", "batch_size",
        "optimizer", "weight_decay", "dropout", "best_val_acc", "epochs_ran"
    ]]
)

Total completed CNN runs: 20
Best CNN validation config per dataset+architecture:


,dataset,architecture,lr,batch_size,optimizer,weight_decay,dropout,best_val_acc,epochs_ran
0,cifar10,enhanced_cnn,0.010,32,adam,0.0000,0.2,0.7316,20
1,cifar10,simple_cnn,0.001,32,adam,0.0001,0.2,0.4656,20


In [ ]:
# Retrain best CNN configs on full train+validation, then evaluate on test
criterion = nn.CrossEntropyLoss()
cnn_retrained_models = {}
cnn_test_rows = []

for _, row in cnn_best_cfg_df.iterrows():
    dataset_name = row["dataset"]
    arch_name = row["architecture"]
    cfg = {
        "lr": float(row["lr"]),
        "batch_size": int(row["batch_size"]),
        "optimizer": str(row["optimizer"]),
        "weight_decay": float(row["weight_decay"]),
        "dropout": float(row["dropout"]),
    }

    train_loader, val_loader, test_loader = get_loaders(dataset_name, cfg["batch_size"])
    full_train_dataset = ConcatDataset([train_loader.dataset, val_loader.dataset])
    full_train_loader = DataLoader(
        full_train_dataset,
        batch_size=cfg["batch_size"],
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    model = build_cnn(dataset_name, arch_name, dropout=cfg["dropout"])
    retrain_cfg = {
        "lr": cfg["lr"],
        "optimizer": cfg["optimizer"],
        "weight_decay": cfg["weight_decay"],
        "epochs": EPOCHS,
    }

    retrained_model, retrain_history, retrain_runtime_sec = train_model(
        model=model,
        train_loader=full_train_loader,
        val_loader=full_train_loader,
        config=retrain_cfg,
        device=device,
    )

    test_loss, test_acc = evaluate(retrained_model, test_loader, criterion, device)
    cnn_retrained_models[(dataset_name, arch_name)] = retrained_model

    cnn_test_rows.append({
        "dataset": dataset_name,
        "architecture": arch_name,
        "lr": cfg["lr"],
        "batch_size": cfg["batch_size"],
        "optimizer": cfg["optimizer"],
        "weight_decay": cfg["weight_decay"],
        "dropout": cfg["dropout"],
        "best_val_acc": float(row["best_val_acc"]),
        "retrain_epochs": len(retrain_history["train_loss"]),
        "retrain_runtime_sec": retrain_runtime_sec,
        "test_loss": test_loss,
        "test_acc": test_acc,
    })

cnn_test_df = pd.DataFrame(cnn_test_rows).sort_values(["dataset", "architecture"]).reset_index(drop=True)
print("Final CNN test performance after full train+validation retraining:")
display(cnn_test_df)

Epoch 1/20 | train_loss=1.5057 | val_loss=1.3375 | val_acc=0.5144
Epoch 2/20 | train_loss=1.2166 | val_loss=1.1138 | val_acc=0.6023
Epoch 3/20 | train_loss=1.1009 | val_loss=0.9483 | val_acc=0.6686
Epoch 4/20 | train_loss=1.0250 | val_loss=0.8571 | val_acc=0.7063
Epoch 5/20 | train_loss=0.9696 | val_loss=0.8416 | val_acc=0.7129
Epoch 6/20 | train_loss=0.9283 | val_loss=0.7865 | val_acc=0.7274
Epoch 7/20 | train_loss=0.8975 | val_loss=0.7471 | val_acc=0.7418
Epoch 8/20 | train_loss=0.8717 | val_loss=0.7136 | val_acc=0.7485
Epoch 9/20 | train_loss=0.8478 | val_loss=0.7183 | val_acc=0.7490
Epoch 10/20 | train_loss=0.8308 | val_loss=0.7029 | val_acc=0.7565
Epoch 11/20 | train_loss=0.8153 | val_loss=0.6836 | val_acc=0.7603
Epoch 12/20 | train_loss=0.7934 | val_loss=0.6356 | val_acc=0.7806
Epoch 13/20 | train_loss=0.7873 | val_loss=0.6250 | val_acc=0.7849
Epoch 14/20 | train_loss=0.7706 | val_loss=0.6162 | val_acc=0.7924
Epoch 15/20 | train_loss=0.7628 | val_loss=0.6153 | val_acc=0.7864
Epoc

,dataset,architecture,lr,batch_size,optimizer,weight_decay,dropout,best_val_acc,retrain_epochs,retrain_runtime_sec,test_loss,test_acc
0,cifar10,enhanced_cnn,0.010,32,adam,0.0000,0.2,0.7316,20,599.187308,0.730787,0.7441
1,cifar10,simple_cnn,0.001,32,adam,0.0001,0.2,0.4656,20,558.086865,1.604734,0.4224


In [10]:
# Save CNN artifacts
os.makedirs("results", exist_ok=True)
results_path = os.path.join("results", "cnn_all_tuning_results.csv")
best_cfg_path = os.path.join("results", "cnn_best_validation_configs.csv")
test_path = os.path.join("results", "cnn_best_models_test_results.csv")

cnn_results_df.to_csv(results_path, index=False)
cnn_best_cfg_df.to_csv(best_cfg_path, index=False)
cnn_test_df.to_csv(test_path, index=False)

print("Saved:")
print(f"- {results_path}")
print(f"- {best_cfg_path}")
print(f"- {test_path}")

Saved:
- results/cnn_all_tuning_results.csv
- results/cnn_best_validation_configs.csv
- results/cnn_best_models_test_results.csv
